In [8]:
import re
import pandas as pd
from tqdm import tqdm

In [3]:
fd = open('big.txt', 'r')
txt = fd.read()
fd.close()

In [4]:
txt[:100]

'The Project Gutenberg EBook of The Adventures of Sherlock Holmes\nby Sir Arthur Conan Doyle\n(#15 in o'

In [5]:
len(txt)

6488666

### 1Finding the Unique Words

In [9]:
with open('big.txt', 'r') as fd:
    lines = fd.readlines()
    words = []
    for line in lines:
        words += re.findall(r'\w+', line.lower())

print(len(words))
vocab = list(set(words))
print(len(vocab))

1115585
32198


### 2 Finding the Probability Distribution

In [10]:
word_probability = {}

for word in tqdm(vocab):
    word_probability[word] = float(words.count(word) / len(words))

100%|██████████| 32198/32198 [07:43<00:00, 69.49it/s]


### 3 Text Preprocessing

#### 3.1 SPlitting Words

In [11]:
def split(word):  
    parts = []
    for i in range(len(word) + 1):
        parts += [(word[:i], word[i:])]
    return parts

### 3.2 2. Deletion

In [14]:
def delete(word):
    output = []
    for l, r in split(word):
        output.append(l + r[1:])
    return output

delete('loave')

['oave', 'lave', 'love', 'loae', 'loav', 'loave']

### 3.3  3 Swapping:

In [15]:
def swap(word):
    output = []    
    for l, r in split(word):
        if len(r) > 1:
            output.append(l + r[1] + r[0] + r[2:])
    return output

swap('lvoe')

['vloe', 'love', 'lveo']

### 3.4  Replacement

In [16]:
def replace(word):
    characters = 'abcdefghijklmnopqrstuvwxyz'
    output = []    
    for l, r in split(word):
        for char in characters:
            output.append(l + char + r[1:])
    return output

len(replace('lave'))

130

### 3.5   Insertion

In [17]:
def insert(word):
    characters = 'abcdefghijklmnopqrstuvwxyz'
    output = []
    for l, r in split(word):
        for char in characters:
            output.append(l + char + r)
    return output

len(insert('lve'))

104

### 4 Finding the Prediction (Level - 1)

### 4.1. Combining Possible Words

In [13]:
def edit(word):   
    return list(set(insert(word) + delete(word) + swap(word) + replace(word)))

### 4.2 2. Predicting the Word

In [18]:
def spell_check_edit_1(word, count=5):
    output = []
    suggested_words = edit(word)
    
    for wrd in suggested_words:        
        if wrd in word_probability.keys():
            output.append([wrd, word_probability[wrd]])
            
    return list(pd.DataFrame(output, columns=['word','prob'])
                .sort_values(by='prob', ascending=False)
                .head(count)['word'].values)

spell_check_edit_1('famili')

['family']

### 5 Finding the Prediction (Level - 2)

In [19]:
def spell_check_edit_2(word, count=5):
    output = []
    suggested_words = edit(word)       
    
    for e1 in edit(word):
        suggested_words += edit(e1)    
    
    suggested_words = list(set(suggested_words))
    
    for wrd in suggested_words:
        if wrd in word_probability.keys():
            output.append([wrd, word_probability[wrd]])
            
    return list(pd.DataFrame(output, columns=['word','prob'])
                .sort_values(by='prob', ascending=False)
                .head(count)['word'].values)

spell_check_edit_2('fameli')

['family', 'namely', 'fame', 'camelia', 'camel']